# 小波变换：复因果递推与 Morlet

本单元定义核配置、状态初始化、逐点更新和单段变换。输入是上游已经确定尺度的实数序列，本单元不取 log、不差分、不标准化。周期 `period_samples` 表示多少个采样点完成一周，时间戳只用于下游对齐，不改变滤波步长。

`kernel_type='causal_gammatone'` 默认采用复 Gammatone 的一阶导数，逐点仅使用当前及历史输入；`kernel_type='morlet'` 提供双边复 Morlet，用于离线窗口比较。两者的周期映射、归一化、导数约定和边界状态随核一起返回，不能用切换名称的方式混用公式。SST 计算归后续 c02，完整序列调度与绘图归后续 c03。

## 函数入口

|函数|输入与返回|
|---|---|
|`build_wavelet_bank(periods, *, kernel_type, kernel_params, warmup_samples, memory_tolerance)`|周期列表与核参数；返回固定核组及逐通道配置|
|`evaluate_wavelet_response(bank, frequencies)`|每采样点的周频率；返回离散滤波器复响应，用于核对频率与零均值|
|`initialize_wavelet_state(bank, initial_value)`|首个有效值；返回恒定背景的稳态，不消耗采样点|
|`update_wavelet_state(state, value)`|一个实数输入；返回当前点结果与新状态，传入状态不被修改|
|`compute_wavelet_transform(values, bank, *, initial_state, extension_mode)`|一段序列；返回按输入位置×通道排列的复系数、导数、状态及可续算状态|

单点更新及状态初始化只适用于因果核。批量因果计算直接复用单点函数；参数中没有窗口重置选项。正常分块、跨日和训练／测试划分通过传回的状态续算。缺失点保留位置，NaN／Inf 中断有效段，下一有效点重新初始化；预热不足与计算失败分别记录。

In [1]:
from pathlib import Path
import sys
project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")


## 因果核、离散化与导数

下式定义卷积核，而非未说明方向的相关运算：

$$g(u)=\frac{u^{q-1}e^{-p u}}{(q-1)!}\mathbf{1}_{u\ge0},\quad
p=\alpha-i\omega_0,\quad h(u)=C g'(u),\quad h_a(u)=a^{-1/2}h(u/a).$$

`q=4, alpha=1, omega0=6` 为默认核参数，`q>=3`、`alpha>0`、`omega0>0`。`C` 将连续母核的 L² 范数归一为 1。`g'` 的积分为零，避免把具有非零均值的调制低通直接称为小波。卷积形式为 `W_a[n]=(x*h_a)[n]`；若写成标准 CWT 相关式，分析基函数为 `psi(u)=conj(h(-u))`，权重位于当前点及其左侧。

导数构造及稳定有理系统依据 Venkitaraman、Adiga、Seelamantula，*Auditory-motivated Gammatone wavelet transform*，Signal Processing 94，608–619，2014，§4。[出版信息](https://doi.org/10.1016/j.sigpro.2013.07.029)；[作者公开全文](https://www.researchgate.net/publication/262347025_Auditory-motivated_Gammatone_wavelet_transform)。本文采用一阶导数，以下离散级联、预畸变映射及状态接口为本实现的明确选择。

连续传递函数为
$$H_a(s)=C a^{3/2-q}\frac{s}{(s+p/a)^q}.$$
采用单位采样间隔的双线性变换 `s=2(1-z⁻¹)/(1+z⁻¹)`，将 `q` 个 `L_a(s)=(p/a)/(s+p/a)` 串联。令 `y_j=L_a^j x`，则
$$W_a=C\frac{a^{3/2}}{p^q}\frac{p}{a}(y_{q-1}-y_q),$$
$$D_a=C\frac{a^{3/2}}{p^q}\left(\frac{p}{a}\right)^2(y_{q-2}-2y_{q-1}+y_q).$$
两者共享当前及历史状态，没有中心差分或未来系数。`D_a` 是双线性模型中的导数；对离散复指数，`D_a/W_a=i·2tan(pi f)`，后续频率估计必须使用逆映射 `f=atan(Im(D/W)/2)/pi`，不能直接除以 `2pi`。[双线性变换定义](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.bilinear.html)

母核正频率幅值峰为
$$\Omega_c=\frac{(q-2)\omega_0+\sqrt{q^2\omega_0^2+4(q-1)\alpha^2}}{2(q-1)}.$$
给定周期 `P>2`，设置 `a=Omega_c/[2tan(pi/P)]`，使实际离散响应峰对应 `1/P`。尺度积分采用相邻尺度间的梯形求积权重；只有一个尺度时权重为空，后续 SST 应要求至少两个尺度。

复因果核是近似解析核，负频率能量不严格为零。输出提供该比例诊断及导数约定；提供 SST 所需接口不等于已经验证 SST 重构或脊线估计。

In [2]:
from dataclasses import dataclass
import hashlib
import json
import math
import numpy as np
import pandas as pd
import pyarrow as pa
from scipy import integrate, optimize, signal, special, stats
from config.data_contracts import pandas_to_arrow

WAVELET_TRANSFORM_VERSION = "0.1.0"

@dataclass(frozen=True)
class WaveletBank:
    kernel_type: str
    parameters: dict
    channels_df: pd.DataFrame
    feed: np.ndarray | None
    pole: np.ndarray | None
    output_multiplier: np.ndarray | None
    fir_kernels: tuple
    fir_derivatives: tuple
    signature: str

@dataclass(frozen=True)
class WaveletState:
    bank: WaveletBank
    filter_values: np.ndarray
    previous_value: float
    segment_id: int
    segment_sample_count: int
    connected: bool

def build_wavelet_bank(periods, *, kernel_type="causal_gammatone", kernel_params=None,
                       warmup_samples=None, memory_tolerance=1e-6) -> WaveletBank:
    """固定周期核组。周期严格递增且 >2；参数错误报错，不裁剪或自动换核。

    warmup_samples 为非负整数或与周期等长的整数列表；None 使用逐通道有效记忆。
    因果记忆来自重复极点的绝对值包络尾质量；不声称某种未知信号已完全稳态。
    Morlet 不使用递推预热，只按完整支持检查左右边界。
    """
    if np.iscomplexobj(periods):
        raise ValueError("periods 必须为实数")
    periods = np.asarray(periods, dtype=float)
    if (periods.ndim != 1 or periods.size == 0 or not np.isfinite(periods).all()
            or (periods <= 2).any() or (np.diff(periods) <= 0).any()):
        raise ValueError("periods 必须为严格递增、有限且大于 2 的一维周期列表")
    if not np.isfinite(memory_tolerance) or not 0 < memory_tolerance < 1:
        raise ValueError("memory_tolerance 必须在 (0, 1) 内")
    supplied = {} if kernel_params is None else dict(kernel_params)
    if kernel_type == "causal_gammatone":
        params = dict(order=4, alpha=1.0, omega0=6.0)
    elif kernel_type == "morlet":
        params = dict(sigma=1.0, omega0=6.0, support_sigma=8.0)
        if warmup_samples is not None:
            raise ValueError("Morlet 无递推预热；请用 extension_mode 定义边界")
    else:
        raise ValueError("kernel_type 必须为 causal_gammatone 或 morlet")
    if supplied.keys() - params.keys():
        raise ValueError(f"不支持的核参数：{sorted(supplied.keys() - params.keys())}")
    params.update(supplied)
    if not np.isfinite(params['omega0']) or params['omega0'] <= 0:
        raise ValueError("omega0 必须为有限正数")
    omega0 = float(params['omega0'])
    kernels, derivatives = [], []
    if kernel_type == "causal_gammatone":
        order = params['order']
        if isinstance(order, (bool, np.bool_)) or not isinstance(order, (int, np.integer)) or order < 3:
            raise ValueError("order 必须为 >=3 的整数，以保证系数及其导数有有限能量")
        alpha = float(params['alpha'])
        if not np.isfinite(alpha) or alpha <= 0:
            raise ValueError("alpha 必须为有限正数")
        params = dict(order=int(order), alpha=alpha, omega0=omega0)
        p = alpha - 1j * omega0
        # 三项母核能量积分；log-gamma 避免阶乘中间量溢出。
        denominator_log = 2 * special.gammaln(order)
        energy_terms = [
            (order-1)**2 * np.exp(special.gammaln(2*order-3) - denominator_log - (2*order-3)*np.log(2*alpha)),
            -2*(order-1)*alpha * np.exp(special.gammaln(2*order-2) - denominator_log - (2*order-2)*np.log(2*alpha)),
            abs(p)**2 * np.exp(special.gammaln(2*order-1) - denominator_log - (2*order-1)*np.log(2*alpha))]
        mother_energy = math.fsum(energy_terms)
        if not np.isfinite(mother_energy) or mother_energy <= 0:
            raise ValueError("核参数导致不可计算的母核能量")
        normalization = 1 / np.sqrt(mother_energy)
        peak = ((order-2)*omega0 + np.sqrt(order**2*omega0**2 + 4*(order-1)*alpha**2)) / (2*(order-1))
        scales = peak / (2*np.tan(np.pi/periods))
        pole = p / scales
        feed = pole / (2+pole)
        discrete_poles = (2-pole)/(2+pole)
        if not np.isfinite(discrete_poles).all() or (abs(discrete_poles) >= 1).any():
            raise ValueError("核参数或周期在当前数值精度下不能生成稳定递推")
        output_multiplier = normalization * scales**1.5 / p**order
        if not np.isfinite(output_multiplier).all():
            raise ValueError("核归一化系数不可计算")
        # 分母重复极点的归一化绝对包络为负二项分布；分子最多再平移 q 点。
        memory = stats.nbinom.ppf(1-memory_tolerance, order, 1-abs(discrete_poles)) + order + 2
        if not np.isfinite(memory).all() or (memory >= np.iinfo(np.int64).max).any():
            raise ValueError("有效记忆长度在当前参数下不可计算")
        memory = np.ceil(memory).astype(np.int64)
        group_delay = order * scales * alpha / (alpha**2+(peak-omega0)**2) / np.cos(np.pi/periods)**2
        # 对无量纲频率积分，避免极大或极小带宽影响数值积分的区间发现。
        ratio = omega0/alpha
        spectrum = lambda v: v*v / (1+(v-ratio)**2)**order
        positive_energy = integrate.quad(spectrum, 0, ratio, epsabs=1e-12)[0] + integrate.quad(spectrum, ratio, np.inf, epsabs=1e-12)[0]
        negative_energy = integrate.quad(lambda v: spectrum(-v), 0, np.inf, epsabs=1e-12)[0]
        leakage = negative_energy/(positive_energy+negative_energy)
        mapping = "bilinear_prewarped_peak"
        derivative_convention = "bilinear_model_derivative"
        normalization_name = "continuous_mother_l2_bilinear"
        support_future = np.zeros(len(periods), dtype=np.int64)
    else:
        sigma, support_sigma = float(params['sigma']), float(params['support_sigma'])
        if not np.isfinite([sigma, support_sigma]).all() or sigma <= 0 or support_sigma <= 0:
            raise ValueError("sigma、support_sigma 必须为有限正数")
        params = dict(sigma=sigma, omega0=omega0, support_sigma=support_sigma)
        correction = np.exp(-0.5*(sigma*omega0)**2)
        mother_energy = np.sqrt(np.pi)*sigma*(1+correction**2-2*correction*np.exp(-0.25*(sigma*omega0)**2))
        if not np.isfinite(mother_energy) or mother_energy <= 0:
            raise ValueError("Morlet 参数导致不可计算的母核能量")
        normalization = 1/np.sqrt(mother_energy)
        amplitude = lambda w: np.exp(-0.5*sigma**2*(w-omega0)**2)-correction*np.exp(-0.5*sigma**2*w*w)
        peak = optimize.minimize_scalar(lambda w: -amplitude(w), bounds=(omega0, omega0+4/sigma), method='bounded').x
        scales = peak*periods/(2*np.pi)
        memory = np.ceil(support_sigma*sigma*scales).astype(np.int64)
        for scale, radius in zip(scales, memory):
            lag = np.arange(-radius, radius+1, dtype=float)
            u = lag/scale
            envelope = np.exp(-0.5*(u/sigma)**2)
            carrier = np.exp(1j*omega0*u)
            # 采样并截断后再扣除离散 DC；记录这一离散近似，不能称为精确连续核。
            discrete_correction = np.sum(envelope*carrier)/np.sum(envelope)
            kernel = normalization/np.sqrt(scale)*envelope*(carrier-discrete_correction)
            derivative = normalization/scale**1.5*envelope*(1j*omega0*carrier-u/sigma**2*(carrier-discrete_correction))
            kernels.append(kernel)
            derivatives.append(derivative)
        group_delay = np.zeros(len(periods))
        positive_energy = integrate.quad(lambda w: amplitude(w)**2, 0, omega0, epsabs=1e-12)[0] + integrate.quad(lambda w: amplitude(w)**2, omega0, np.inf, epsabs=1e-12)[0]
        negative_energy = integrate.quad(lambda w: amplitude(-w)**2, 0, np.inf, epsabs=1e-12)[0]
        leakage = negative_energy/(positive_energy+negative_energy)
        feed = pole = output_multiplier = None
        mapping = "continuous_peak_sampled_fir"
        derivative_convention = "sampled_continuous_kernel_derivative"
        normalization_name = "continuous_mother_l2_sampled_fir_dc_corrected"
        support_future = memory.copy()
    if warmup_samples is None:
        warmup = memory.copy() if kernel_type == 'causal_gammatone' else np.zeros(len(periods), dtype=np.int64)
    else:
        requested = np.asarray(warmup_samples)
        if requested.ndim == 0:
            requested = np.repeat(requested, len(periods))
        if (requested.shape != periods.shape or requested.dtype.kind not in 'iu'
                or (requested < 0).any() or (requested >= np.iinfo(np.int64).max).any()):
            raise ValueError("warmup_samples 必须为非负整数或与周期等长的非负整数列表")
        warmup = requested.astype(np.int64)
    widths = np.full(len(scales), np.nan)
    if len(scales) >= 2:
        widths[0], widths[-1] = np.diff(scales)[0]/2, np.diff(scales)[-1]/2
        if len(scales) > 2:
            widths[1:-1] = (scales[2:]-scales[:-2])/2
    channels_df = pd.DataFrame(dict(channel_id=np.arange(len(periods)), period_samples=periods,
        frequency_cycles_per_sample=1/periods, scale=scales, scale_weight=widths*scales**(-1.5),
        warmup_samples=warmup, effective_memory_samples=memory, support_future_samples=support_future,
        group_delay_samples=group_delay, negative_frequency_energy_ratio=np.repeat(leakage, len(periods))))
    parameters = dict(kernel_params=params, memory_tolerance=float(memory_tolerance),
        normalization=normalization_name, mother_normalization=float(normalization), mother_peak_angular_frequency=float(peak),
        period_mapping=mapping, derivative_convention=derivative_convention,
        is_causal=kernel_type=='causal_gammatone', supports_sst=True, exact_analytic=False,
        warmup_samples=warmup.tolist())
    signature = hashlib.sha256(json.dumps(dict(version=WAVELET_TRANSFORM_VERSION, kernel_type=kernel_type,
        periods=periods.tolist(), parameters=parameters), sort_keys=True, allow_nan=False).encode()).hexdigest()
    for array in [feed, pole, output_multiplier, *kernels, *derivatives]:
        if array is not None:
            array.setflags(write=False)
    return WaveletBank(kernel_type, parameters, channels_df, feed, pole, output_multiplier,
        tuple(kernels), tuple(derivatives), signature)

def evaluate_wavelet_response(bank: WaveletBank, frequencies) -> np.ndarray:
    """离散周频率 [-0.5,0.5]，返回 frequency×channel 复响应；不估计输入谱。"""
    if np.iscomplexobj(frequencies):
        raise ValueError("frequencies 必须为实数")
    frequencies = np.asarray(frequencies, dtype=float)
    if frequencies.ndim != 1 or not np.isfinite(frequencies).all() or (abs(frequencies)>0.5).any():
        raise ValueError("frequencies 必须为 [-0.5, 0.5] 的有限一维数组")
    if bank.kernel_type == 'causal_gammatone':
        # z 多项式形式在 Nyquist 处也有定义，避免 tan(pi/2) 的数值奇点。
        z = np.exp(-2j*np.pi*frequencies[:,None])
        p = bank.pole[None,:]
        q = bank.parameters['kernel_params']['order']
        r = (2-p)/(2+p)
        response = bank.output_multiplier[None,:]*p**q*2*(1-z)*(1+z)**(q-1)/(2+p)**q/(1-r*z)**q
    else:
        response = np.empty((len(frequencies), len(bank.fir_kernels)), dtype=complex)
        for j, kernel in enumerate(bank.fir_kernels):
            radius = (len(kernel)-1)//2
            response[:,j] = signal.freqz(kernel, [1], worN=2*np.pi*frequencies)[1]*np.exp(2j*np.pi*frequencies*radius)
    return response


## 初始状态、预热及失败

有效段开始时将全部低通级的状态设为首个有效输入值，表示此前是同值恒定背景；初始小波及导数为零。初始化不假定已经有足够真实历史。

默认预热长度由逐通道重复极点的绝对值包络尾质量决定：以 `|r|` 建立负二项分布，取 `1-memory_tolerance` 分位数，再计入有限分子平移。它是保守的有效记忆定义，不是固定窗口截断，滤波状态始终连续递推。前 `warmup_samples` 个有效点为 `warmup`，其系数与导数置空；调用者可以指定各周期预热长度，包括 0。

非有限输入点记录 `failed/nonfinite_input`，不填充、不删除；下一有效点开启新有效段。数值溢出同样保留失败并中断状态，不自动换核或重试。正常市场休市不产生额外采样点，也不触发重置。

Morlet 使用高斯包络截断支持；`extension_mode='none'` 默认将支持不完整的位置标为 `boundary_incomplete`。`constant`、`reflect`、`symmetric`、`periodic` 分别显式定义边界延拓；它们改变离线窗口估计。因果分支仅接受 None／`none`，不允许右侧延拓。

In [3]:
def initialize_wavelet_state(bank: WaveletBank, initial_value) -> WaveletState:
    """恒定背景稳态；不消耗输入点，只支持因果核。"""
    if bank.kernel_type != 'causal_gammatone':
        raise ValueError("状态递推只支持 causal_gammatone")
    if np.iscomplexobj(initial_value) or not np.isscalar(initial_value) or not np.isfinite(initial_value):
        raise ValueError("initial_value 必须为有限实数")
    order = bank.parameters['kernel_params']['order']
    filters = np.full((len(bank.channels_df), order), float(initial_value), dtype=complex)
    filters.setflags(write=False)
    return WaveletState(bank, filters, float(initial_value), 0, 0, True)

def update_wavelet_state(state: WaveletState, value):
    """返回 (point, new_state)，不修改传入状态。新状态适用于下一采样点。"""
    bank = state.bank
    if bank.kernel_type != 'causal_gammatone':
        raise ValueError("状态递推只支持 causal_gammatone")
    if np.iscomplexobj(value) or not np.isscalar(value):
        raise ValueError("value 必须为实数标量；缺失用 NaN 表示")
    value = float(value)
    channels = len(bank.channels_df)
    coefficients = np.full(channels, complex(np.nan, np.nan))
    derivatives = coefficients.copy()
    if not np.isfinite(value):
        status = np.full(channels, 'failed', dtype=object)
        reason = np.full(channels, 'nonfinite_input', dtype=object)
        new_state = WaveletState(bank, state.filter_values, state.previous_value, state.segment_id, 0, False)
        count, segment_id = 0, state.segment_id
    else:
        if state.connected:
            previous_filters, previous_value = state.filter_values, state.previous_value
            segment_id, count = state.segment_id, state.segment_sample_count+1
        else:
            initialized = initialize_wavelet_state(bank, value)
            previous_filters, previous_value = initialized.filter_values, initialized.previous_value
            segment_id, count = state.segment_id+1, 1
        current_filters = np.empty_like(previous_filters)
        with np.errstate(over='ignore', invalid='ignore'):
            for j in range(current_filters.shape[1]):
                current_input = value if j==0 else current_filters[:,j-1]
                previous_input = previous_value if j==0 else previous_filters[:,j-1]
                previous_output = previous_filters[:,j]
                # 等价于 r*y[-1]+feed*(u+u[-1])，差值形式保持恒定背景精确为零。
                current_filters[:,j] = previous_output + bank.feed*((current_input-previous_output)+(previous_input-previous_output))
            coefficients = bank.output_multiplier*bank.pole*(current_filters[:,-2]-current_filters[:,-1])
            derivatives = bank.output_multiplier*bank.pole**2*(current_filters[:,-3]-2*current_filters[:,-2]+current_filters[:,-1])
        numerical_good = np.isfinite(coefficients)&np.isfinite(derivatives)&np.isfinite(current_filters).all(axis=1)
        status = np.where(count>bank.channels_df['warmup_samples'].to_numpy(), 'ok', 'warmup').astype(object)
        reason = np.where(status=='ok', '', 'insufficient_history').astype(object)
        if not numerical_good.all():
            # 任一递推级失效即中断这条输入序列的状态；不得将无效内部状态带到下一点。
            status[:] = 'failed'
            reason[:] = 'nonfinite_filter_state'
            current_filters = previous_filters.copy()
        unavailable = status!='ok'
        coefficients[unavailable], derivatives[unavailable] = complex(np.nan, np.nan), complex(np.nan, np.nan)
        current_filters.setflags(write=False)
        new_state = WaveletState(bank, current_filters, value, segment_id, count, bool(numerical_good.all()))
    point = dict(coefficients=coefficients, derivatives=derivatives, status=status, reason=reason,
        segment_id=segment_id, segment_sample_count=count)
    return point, new_state

def compute_wavelet_transform(values, bank: WaveletBank, *, initial_state=None, extension_mode=None) -> dict:
    """time×channel 输出。输入顺序就是采样顺序，失败保留；不取时间差、不绘图。

    因果分支只接受 none，final_state 可用于分块续算。Morlet 每次是一段离线窗口，
    不支持 initial_state；NaN/Inf 两侧各为独立有限段，不跨缺失点卷积。
    """
    if np.iscomplexobj(values):
        raise ValueError("values 必须为实数序列")
    values = np.asarray(values, dtype=float)
    if values.ndim != 1:
        raise ValueError("values 必须为一维序列")
    mode = 'none' if extension_mode is None else extension_mode
    allowed = ('none', 'constant', 'reflect', 'symmetric', 'periodic')
    if mode not in allowed:
        raise ValueError(f"extension_mode 必须为 {allowed}")
    length, channels = len(values), len(bank.channels_df)
    coefficients = np.full((length, channels), complex(np.nan,np.nan))
    derivatives = coefficients.copy()
    status = np.full((length,channels), 'failed', dtype=object)
    reason = np.full((length,channels), 'nonfinite_input', dtype=object)
    segment_ids = np.full(length,-1,dtype=np.int64)
    segment_counts = np.zeros(length,dtype=np.int64)
    final_state = None
    if bank.kernel_type == 'causal_gammatone':
        if mode != 'none':
            raise ValueError("因果核不允许边界延拓")
        if initial_state is not None and initial_state.bank.signature != bank.signature:
            raise ValueError("initial_state 与核组身份不一致")
        order = bank.parameters['kernel_params']['order']
        state = initial_state
        if state is None:
            filters = np.zeros((channels,order),dtype=complex)
            filters.setflags(write=False)
            state = WaveletState(bank,filters,0.0,-1,0,False)
        for i,value in enumerate(values):
            point,state = update_wavelet_state(state,value)
            coefficients[i],derivatives[i] = point['coefficients'],point['derivatives']
            status[i],reason[i] = point['status'],point['reason']
            segment_ids[i],segment_counts[i] = point['segment_id'],point['segment_sample_count']
        final_state = state
    else:
        if initial_state is not None:
            raise ValueError("Morlet 离线变换不接受递推状态")
        finite = np.isfinite(values)
        starts = np.flatnonzero(finite & ~np.r_[False,finite[:-1]]) if length else np.array([],dtype=int)
        stops = np.flatnonzero(finite & ~np.r_[finite[1:],False])+1 if length else np.array([],dtype=int)
        for segment_id,(start,stop) in enumerate(zip(starts,stops)):
            segment = values[start:stop]
            segment_ids[start:stop] = segment_id
            segment_counts[start:stop] = np.arange(1,len(segment)+1)
            for j,(kernel,derivative) in enumerate(zip(bank.fir_kernels,bank.fir_derivatives)):
                radius = (len(kernel)-1)//2
                if mode == 'none':
                    extended,offset = segment,radius
                    available = (np.arange(len(segment))>=radius)&(np.arange(len(segment))+radius<len(segment))
                else:
                    pad_mode = dict(constant='edge',reflect='reflect',symmetric='symmetric',periodic='wrap')[mode]
                    extended,offset = np.pad(segment,radius,mode=pad_mode),2*radius
                    available = np.ones(len(segment),dtype=bool)
                with np.errstate(over='ignore',invalid='ignore'):
                    channel_coefficients = signal.fftconvolve(extended,kernel,mode='full')[offset:offset+len(segment)]
                    channel_derivatives = signal.fftconvolve(extended,derivative,mode='full')[offset:offset+len(segment)]
                numerical_good = np.isfinite(channel_coefficients)&np.isfinite(channel_derivatives)
                channel_status = np.where(available,'ok','boundary_incomplete').astype(object)
                channel_reason = np.where(available,'','incomplete_kernel_support').astype(object)
                channel_status[~numerical_good],channel_reason[~numerical_good] = 'failed','nonfinite_convolution'
                good = channel_status=='ok'
                coefficients[start:stop,j] = np.where(good,channel_coefficients,complex(np.nan,np.nan))
                derivatives[start:stop,j] = np.where(good,channel_derivatives,complex(np.nan,np.nan))
                status[start:stop,j],reason[start:stop,j] = channel_status,channel_reason
    return dict(bank=bank,coefficients=coefficients,derivatives=derivatives,status=status,reason=reason,
        segment_id=segment_ids,segment_sample_count=segment_counts,final_state=final_state,extension_mode=mode)


## 结果字段与落盘

计算返回复数矩阵，行数与输入长度一致，列顺序由 `channel_id` 固定。`status == 'ok'` 才有可使用的系数与导数。`warmup` 和 `boundary_incomplete` 是信息不足状态；`failed` 是输入或计算失败。没有把这些位置压缩掉。

demo 长表将复数拆为实部／虚部，幅值为复系数绝对值，能量为幅值平方；零幅值的相位为空。`sample_at` 为采样位置时刻，`bar_at` 为所用完整源分钟的报价可得时刻；本单元不将离线 Morlet 的输出宣称为当时可用的交易特征。预热、未来支持及响应滞后写入通道表，不通过回移时间戳消除滞后。

In [4]:
WAVELET_CHANNEL_SCHEMA = pa.schema([
    pa.field('channel_id',pa.int64()),pa.field('period_samples',pa.float64()),
    pa.field('frequency_cycles_per_sample',pa.float64()),pa.field('scale',pa.float64()),
    pa.field('scale_weight',pa.float64()),pa.field('warmup_samples',pa.int64()),
    pa.field('effective_memory_samples',pa.int64()),pa.field('support_future_samples',pa.int64()),
    pa.field('group_delay_samples',pa.float64()),pa.field('negative_frequency_energy_ratio',pa.float64()),
],metadata={b'method':b'wavelet_transform',b'schema_version':b'0.1.0',b'period_unit':b'samples'})

WAVELET_COEFFICIENT_SCHEMA = pa.schema([
    pa.field('sample_id',pa.int64()),pa.field('sample_at',pa.timestamp('us',tz='Asia/Shanghai')),
    pa.field('bar_at',pa.timestamp('us',tz='Asia/Shanghai')),pa.field('trading_date',pa.date32()),
    pa.field('sample_side',pa.string()),pa.field('input_value',pa.float64()),pa.field('channel_id',pa.int64()),
    pa.field('coefficient_real',pa.float64()),pa.field('coefficient_imag',pa.float64()),
    pa.field('derivative_real',pa.float64()),pa.field('derivative_imag',pa.float64()),
    pa.field('amplitude',pa.float64()),pa.field('energy',pa.float64()),pa.field('phase',pa.float64()),
    pa.field('segment_id',pa.int64()),pa.field('segment_sample_count',pa.int64()),
    pa.field('transform_status',pa.string()),pa.field('transform_reason',pa.string()),
],metadata={b'method':b'wavelet_transform',b'schema_version':b'0.1.0',b'input_transform':b'identity',
    b'period_unit':b'samples',b'coefficient_orientation':b'causal_convolution_or_centered_fir'})


In [5]:
WAVELET_COLUMN_LABELS = {
    'channel_id':'channel_id（通道编号）','period_samples':'period_samples（周期采样点数）',
    'frequency_cycles_per_sample':'frequency_cycles_per_sample（每采样点周频率）',
    'scale':'scale（连续核尺度）','scale_weight':'scale_weight（SST尺度求积权重）',
    'warmup_samples':'warmup_samples（预热采样点数）',
    'effective_memory_samples':'effective_memory_samples（有效记忆采样点数）',
    'support_future_samples':'support_future_samples（所需未来采样点数）',
    'group_delay_samples':'group_delay_samples（峰值频率响应滞后点数）',
    'negative_frequency_energy_ratio':'negative_frequency_energy_ratio（连续核负频能量比例）',
    'input_value':'input_value（上游输入值）','coefficient_real':'coefficient_real（系数实部）',
    'coefficient_imag':'coefficient_imag（系数虚部）','derivative_real':'derivative_real（导数实部）',
    'derivative_imag':'derivative_imag（导数虚部）','amplitude':'amplitude（系数幅值）',
    'energy':'energy（系数能量）','phase':'phase（系数相位弧度）','segment_id':'segment_id（有效段编号）',
    'segment_sample_count':'segment_sample_count（段内有效点数）',
    'transform_status':'transform_status（变换状态）','transform_reason':'transform_reason（状态原因）',
    'sample_side':'sample_side（训练测试侧）','kernel_type':'kernel_type（核类型）',
    'sample_id':'sample_id（采样点编号）','sample_at':'sample_at（采样位置时刻）',
    'bar_at':'bar_at（源报价可得时刻）','trading_date':'trading_date（交易日）',
}


## 真实 demo：完整 RB 主力采样收益

显式读取 b03 已保存的 RB、N=60、实际分钟轴、全合约数量口径、log 插值、固定分界配置；采用上游 `build_main_sample_series` 生成的 `log_return`，小波函数直接使用这个输入值。请求范围为训练 2010-01-01 至 2024-12-31、测试 2025-01-01 至 2026-09-30；完整序列保留最初差分缺失及其他原有缺失。

两种核分别覆盖整个真实序列，周期为 8、16、32、64、128、256 个采样点。因果核使用默认有效记忆预热；Morlet 使用 `extension_mode='none'`，如实记录左右支持不足。下列真实数据核验包括逐点／批量一致、按训练测试边界续算、追加数据不改历史、幅值和能量关系，以及每个对外计算入口的实际调用。合成边界测试另行执行，不能代替本 demo。

2026-10-07 实际验收：21,316 个输入点，首个上游差分缺失；每种核覆盖 127,896 个通道点。因果核成功 117,485、预热 10,405、缺失 6；Morlet 成功 120,184、左右支持不足 7,706、缺失 6。缺失 6 行对应同一输入点的六个通道，无数值计算失败。四份 Parquet 已复读核对 Schema、身份及内容摘要；完整真实序列的分块续算、历史不变性与输出关系验证通过。12 项独立合成数学与边界检查另行通过。SST 及其重构尚未实现。

In [6]:
import os
import platform
import scipy
import pyarrow.parquet as pq
from datetime import datetime,timezone
from IPython.display import display
from R04_Research.a01_Methods.b03_Sampling.c02_SamplingExecution import read_sampling_demo_main_series
from R04_Research.a01_Methods.b02_MainContinuousAdjustment.c01_MainContinuousAdjustment import arrow_content_sha256

demo_main_df,_,demo_upstream_context = read_sampling_demo_main_series(project_root=candidate_root,
    underlying_code='RB',N=60,axis='time',quantity_scope='all_contracts',sampling_method='log_interpolate')
demo_values = demo_main_df['log_return'].to_numpy(dtype=float,na_value=np.nan)
demo_periods = [8,16,32,64,128,256]
demo_producer_notebook = candidate_root/'R04_Research/a01_Methods/b06_WaveletAnalysis/c01_WaveletTransform.ipynb'
demo_output_dir = demo_producer_notebook.parent/'c01_WaveletTransform_DemoData'
demo_range_token = 'train20100101-20241231_test20250101-20260930'
demo_input_token = 'N60_time_all_contracts_log_interpolate_fixed_log_return'
demo_banks = {kind:build_wavelet_bank(demo_periods,kernel_type=kind) for kind in ['causal_gammatone','morlet']}
print(f'真实上游主力序列：{len(demo_values):,} 点；非有限输入 {np.count_nonzero(~np.isfinite(demo_values))}',flush=True)


真实上游主力序列：21,316 点；非有限输入 1


In [7]:
demo_transforms = {}
for demo_kind,demo_bank in demo_banks.items():
    demo_transforms[demo_kind] = compute_wavelet_transform(demo_values,demo_bank,extension_mode='none')
    np.testing.assert_allclose(evaluate_wavelet_response(demo_bank,[0])[0],0,atol=2e-12)
    assert demo_transforms[demo_kind]['coefficients'].shape == (len(demo_values),len(demo_periods))
    print(f"完整 {demo_kind} 计算完成；通道状态数 {pd.Series(demo_transforms[demo_kind]['status'].ravel()).value_counts().to_dict()}",flush=True)

demo_causal_bank = demo_banks['causal_gammatone']
demo_first_finite = np.flatnonzero(np.isfinite(demo_values))[0]
demo_state = initialize_wavelet_state(demo_causal_bank,demo_values[demo_first_finite])
demo_point,demo_state = update_wavelet_state(demo_state,demo_values[demo_first_finite])
np.testing.assert_array_equal(demo_point['status'],demo_transforms['causal_gammatone']['status'][demo_first_finite])
demo_split = int(np.flatnonzero(demo_main_df['sample_side'].eq('test').to_numpy())[0])
demo_left = compute_wavelet_transform(demo_values[:demo_split],demo_causal_bank)
demo_right = compute_wavelet_transform(demo_values[demo_split:],demo_causal_bank,initial_state=demo_left['final_state'])
for demo_field in ['coefficients','derivatives']:
    np.testing.assert_array_equal(np.concatenate([demo_left[demo_field],demo_right[demo_field]]),demo_transforms['causal_gammatone'][demo_field])
    np.testing.assert_array_equal(demo_left[demo_field],demo_transforms['causal_gammatone'][demo_field][:demo_split])
for demo_field in ['status','reason','segment_id','segment_sample_count']:
    np.testing.assert_array_equal(np.concatenate([demo_left[demo_field],demo_right[demo_field]]),demo_transforms['causal_gammatone'][demo_field])

demo_finite_segment = demo_values[demo_first_finite:demo_first_finite+500]
demo_manual_state = initialize_wavelet_state(demo_causal_bank,demo_finite_segment[0])
demo_manual_coefficients = []
for demo_value in demo_finite_segment:
    demo_point,demo_manual_state = update_wavelet_state(demo_manual_state,demo_value)
    demo_manual_coefficients.append(demo_point['coefficients'])
np.testing.assert_array_equal(demo_manual_coefficients,
    compute_wavelet_transform(demo_finite_segment,demo_causal_bank)['coefficients'])
for demo_j,demo_frequency in enumerate(1/np.asarray(demo_periods,dtype=float)):
    demo_peak = optimize.minimize_scalar(lambda f:-abs(evaluate_wavelet_response(demo_causal_bank,[f])[0,demo_j]),
        bounds=(demo_frequency*0.8,min(0.499,demo_frequency*1.2)),method='bounded',options={'xatol':1e-12}).x
    np.testing.assert_allclose(demo_peak,demo_frequency,rtol=2e-6)
print('完整真实序列的分块续算、前缀不变性、独立单点调用及频率映射核验通过',flush=True)


完整 causal_gammatone 计算完成；通道状态数 {'ok': 117485, 'warmup': 10405, 'failed': 6}
完整 morlet 计算完成；通道状态数 {'ok': 120184, 'boundary_incomplete': 7706, 'failed': 6}
完整真实序列的分块续算、前缀不变性、独立单点调用及频率映射核验通过


In [8]:
demo_coefficients = {}
demo_status_tables = []
for demo_kind,demo_transform in demo_transforms.items():
    demo_coef = demo_transform['coefficients'].ravel()
    demo_derivative = demo_transform['derivatives'].ravel()
    demo_amplitude = abs(demo_coef)
    demo_phase = np.where(demo_amplitude>0,np.angle(demo_coef),np.nan)
    demo_df = demo_main_df[['sample_id','sample_at','bar_at','trading_date','sample_side']].iloc[
        np.repeat(np.arange(len(demo_values)),len(demo_periods))].reset_index(drop=True).copy()
    demo_df['input_value'] = np.repeat(demo_values,len(demo_periods))
    demo_df['channel_id'] = np.tile(np.arange(len(demo_periods)),len(demo_values))
    for demo_name,demo_column in [('coefficient_real',demo_coef.real),('coefficient_imag',demo_coef.imag),
            ('derivative_real',demo_derivative.real),('derivative_imag',demo_derivative.imag),
            ('amplitude',demo_amplitude),('energy',demo_amplitude**2),('phase',demo_phase)]:
        demo_df[demo_name] = demo_column
    demo_df['segment_id'] = np.repeat(demo_transform['segment_id'],len(demo_periods))
    demo_df['segment_sample_count'] = np.repeat(demo_transform['segment_sample_count'],len(demo_periods))
    demo_df['transform_status'] = demo_transform['status'].ravel()
    demo_df['transform_reason'] = demo_transform['reason'].ravel()
    demo_good = demo_df['transform_status'].eq('ok')
    assert np.isfinite(demo_df.loc[demo_good,['coefficient_real','coefficient_imag','derivative_real','derivative_imag']]).all().all()
    assert demo_df.loc[~demo_good,['coefficient_real','coefficient_imag','derivative_real','derivative_imag']].isna().all().all()
    np.testing.assert_allclose(demo_df.loc[demo_good,'amplitude'],np.hypot(demo_df.loc[demo_good,'coefficient_real'],demo_df.loc[demo_good,'coefficient_imag']))
    np.testing.assert_allclose(demo_df.loc[demo_good,'energy'],demo_df.loc[demo_good,'amplitude']**2)
    assert demo_df.loc[demo_df['amplitude'].eq(0),'phase'].isna().all()
    demo_missing = np.repeat(~np.isfinite(demo_values),len(demo_periods))
    assert demo_df.loc[demo_missing,'transform_reason'].eq('nonfinite_input').all()
    demo_coefficients[demo_kind] = demo_df
    demo_status_tables.append(demo_df.groupby(['channel_id','transform_status','transform_reason'],dropna=False).size()
        .rename('通道点数').reset_index().assign(kernel_type=demo_kind))


In [9]:
demo_book = json.loads(demo_producer_notebook.read_text(encoding='utf-8'))
demo_method_source = '\n\n'.join(''.join(c['source']) for c in demo_book['cells']
    if c['cell_type']=='code' and 'demo-dependency' in c.get('metadata',{}).get('tags',[]))
demo_upstream_books = {role:candidate_root/f'R04_Research/a01_Methods/b03_Sampling/{name}.ipynb'
    for role,name in [('sampling_positions','c01_SamplingPositions'),('sampling_execution','c02_SamplingExecution')]}
demo_dependencies = {}
for demo_role,demo_path in demo_upstream_books.items():
    demo_dependency_book = json.loads(demo_path.read_text(encoding='utf-8'))
    demo_source = '\n\n'.join(''.join(c['source']) for c in demo_dependency_book['cells']
        if c['cell_type']=='code' and {'export','demo-dependency'}.issubset(c.get('metadata',{}).get('tags',[])))
    demo_dependencies[demo_role] = hashlib.sha256(demo_source.encode()).hexdigest()
demo_base_identity = dict(artifact_role='method_demo',method_version=WAVELET_TRANSFORM_VERSION,
    producer_notebook=demo_producer_notebook.relative_to(candidate_root).as_posix(),method_source_tag='demo-dependency',
    method_source_sha256=hashlib.sha256(demo_method_source.encode()).hexdigest(),
    dependency_method_source_sha256=demo_dependencies,
    dependency_source_sha256={role:hashlib.sha256((candidate_root/relative).read_bytes()).hexdigest()
        for role,relative in [('notebook_loader','R04_Research/a00_notebook_loader.py'),('data_contracts','config/data_contracts.py')]},
    upstream_demo_identity={role:demo_upstream_context[role] for role in ['positions_identity','prices_identity']},
    upstream_paths={role:demo_upstream_context[role] for role in ['positions_path','prices_path']},
    scope=demo_upstream_context['prices_identity']['scope'],timezone='Asia/Shanghai',
    actual_sample_at_start=str(demo_main_df['sample_at'].min()),actual_sample_at_end=str(demo_main_df['sample_at'].max()),
    input_rows=len(demo_values),input_nonfinite_count=int((~np.isfinite(demo_values)).sum()),
    runtime=dict(python=sys.version.split()[0],numpy=np.__version__,pandas=pd.__version__,pyarrow=pa.__version__,
        scipy=scipy.__version__,platform=platform.platform()))
demo_output_dir.mkdir(parents=True,exist_ok=True)
for demo_kind,demo_bank in demo_banks.items():
    demo_kernel_token = 'causal_gammatone_q4_alpha1_omega6_mem1e-6' if demo_kind=='causal_gammatone' else 'morlet_sigma1_omega6_support8_none'
    demo_config_token = f'{demo_kernel_token}_P8-16-32-64-128-256'
    for demo_role,demo_frame,demo_schema,demo_sort_keys in [
        ('channels',demo_bank.channels_df,WAVELET_CHANNEL_SCHEMA,[('channel_id','ascending')]),
        ('coefficients',demo_coefficients[demo_kind],WAVELET_COEFFICIENT_SCHEMA,[('sample_id','ascending'),('channel_id','ascending')])]:
        demo_table = pandas_to_arrow(demo_frame,demo_schema).sort_by(demo_sort_keys)
        demo_identity = dict(demo_base_identity,output_role=demo_role,output_rows=demo_table.num_rows,output_sort_keys=demo_sort_keys,
            output_schema_sha256=hashlib.sha256(demo_schema.serialize().to_pybytes()).hexdigest(),
            output_content_sha256=arrow_content_sha256(demo_table,demo_schema,demo_sort_keys),
            parameters=dict(kernel_type=demo_kind,periods=demo_periods,**demo_bank.parameters,extension_mode='none',
                input_col='log_return',input_transform='identity',upstream_input_transform='global_log_difference',
                sampling=demo_input_token),bank_signature=demo_bank.signature,
            status_counts={str(k):int(v) for k,v in pd.Series(demo_transforms[demo_kind]['status'].ravel()).value_counts().items()})
        demo_identity = json.loads(json.dumps(demo_identity,ensure_ascii=False,allow_nan=False))
        demo_path = demo_output_dir/f'c01_WaveletTransform_RB_{demo_range_token}_{demo_input_token}_{demo_config_token}_{demo_role}_demo.parquet'
        demo_staging_path = demo_path.with_name(demo_path.name+'.staging')
        if demo_staging_path.exists():
            raise RuntimeError(f'存在未处理 staging：{demo_staging_path}')
        demo_preserve = False
        if demo_path.exists():
            demo_existing_schema = pq.read_schema(demo_path)
            demo_existing_identity = json.loads(demo_existing_schema.metadata[b'demo_identity'])
            demo_existing_base = demo_existing_schema.with_metadata({k:v for k,v in demo_existing_schema.metadata.items() if k!=b'demo_identity'})
            assert arrow_content_sha256(demo_path,demo_existing_base,demo_existing_identity['output_sort_keys'])==demo_existing_identity['output_content_sha256']
            demo_preserve = demo_existing_base.equals(demo_schema,check_metadata=True) and {k:v for k,v in demo_existing_identity.items() if k!='generated_at_utc'}==demo_identity
        if not demo_preserve:
            demo_installed_identity = dict(demo_identity,generated_at_utc=datetime.now(timezone.utc).isoformat())
            demo_metadata = dict(demo_schema.metadata)
            demo_metadata[b'demo_identity'] = json.dumps(demo_installed_identity,ensure_ascii=False,allow_nan=False,sort_keys=True).encode()
            pq.write_table(demo_table.replace_schema_metadata(demo_metadata),demo_staging_path,compression='zstd')
            assert arrow_content_sha256(demo_staging_path,demo_schema,demo_sort_keys)==demo_identity['output_content_sha256']
            os.replace(demo_staging_path,demo_path)
        demo_saved_schema = pq.read_schema(demo_path)
        assert demo_saved_schema.with_metadata({k:v for k,v in demo_saved_schema.metadata.items() if k!=b'demo_identity'}).equals(demo_schema,check_metadata=True)
        demo_saved_identity = json.loads(demo_saved_schema.metadata[b'demo_identity'])
        assert {k:v for k,v in demo_saved_identity.items() if k!='generated_at_utc'}==demo_identity
        assert arrow_content_sha256(demo_path,demo_schema,demo_sort_keys)==demo_identity['output_content_sha256']
        print(f'{demo_kind} {demo_role} 已保存并复读验收：{demo_table.num_rows:,} 行',flush=True)


causal_gammatone channels 已保存并复读验收：6 行
causal_gammatone coefficients 已保存并复读验收：127,896 行
morlet channels 已保存并复读验收：6 行
morlet coefficients 已保存并复读验收：127,896 行


In [10]:
for demo_kind,demo_bank in demo_banks.items():
    print(f'核类型：{demo_kind}')
    display(demo_bank.channels_df.rename(columns=WAVELET_COLUMN_LABELS).rename_axis('row（周期配置行）'))
    display(demo_coefficients[demo_kind].tail(8).rename(columns=WAVELET_COLUMN_LABELS).rename_axis('row（末端变换行）'))
display(pd.concat(demo_status_tables,ignore_index=True).rename(columns=WAVELET_COLUMN_LABELS).rename_axis('row（真实状态汇总行）'))
print('c01 真实 demo 已完成；SST 与完整序列调度／绘图分别归后续 c02、c03。')


核类型：causal_gammatone
核类型：morlet
c01 真实 demo 已完成；SST 与完整序列调度／绘图分别归后续 c02、c03。


,channel_id（通道编号）,period_samples（周期采样点数）,frequency_cycles_per_sample（每采样点周频率）,scale（连续核尺度）,scale_weight（SST尺度求积权重）,warmup_samples（预热采样点数）,effective_memory_samples（有效记忆采样点数）,support_future_samples（所需未来采样点数）,group_delay_samples（峰值频率响应滞后点数）,negative_frequency_energy_ratio（连续核负频能量比例）
row（周期配置行）,,,,,,,,,,
0,0,8.0,0.125000,7.292678,0.200406,186,186,0,34.116991,3.292161e-08
1,1,16.0,0.062500,15.186215,0.197510,341,341,0,63.039979,3.292161e-08
2,2,32.0,0.031250,30.669945,0.136302,665,665,0,123.657368,3.292161e-08
3,3,64.0,0.015625,61.488289,0.095799,1320,1320,0,246.123847,3.292161e-08
4,4,128.0,0.007812,123.050733,0.067638,2633,2633,0,491.654760,3.292161e-08
5,5,256.0,0.003906,246.138538,0.015937,5260,5260,0,983.013366,3.292161e-08


,sample_id（采样点编号）,sample_at（采样位置时刻）,bar_at（源报价可得时刻）,trading_date（交易日）,sample_side（训练测试侧）,input_value（上游输入值）,channel_id（通道编号）,coefficient_real（系数实部）,coefficient_imag（系数虚部）,derivative_real（导数实部）,derivative_imag（导数虚部）,amplitude（系数幅值）,energy（系数能量）,phase（系数相位弧度）,segment_id（有效段编号）,segment_sample_count（段内有效点数）,transform_status（变换状态）,transform_reason（状态原因）
row（末端变换行）,,,,,,,,,,,,,,,,,,
127888,21315,2026-09-30 14:00:00+08:00,2026-09-30 14:00:00+08:00,2026-09-30,test,-0.000321,4,0.001275,-0.002276,0.000104,0.000063,0.002609,0.000007,-1.060184,0,21314,ok,
127889,21315,2026-09-30 14:00:00+08:00,2026-09-30 14:00:00+08:00,2026-09-30,test,-0.000321,5,-0.000882,-0.000589,0.000011,-0.000026,0.001061,0.000001,-2.553180,0,21314,ok,
127890,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,0,-0.000784,-0.000689,0.000487,-0.000782,0.001044,0.000001,-2.420197,0,21315,ok,
127891,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,1,-0.002186,0.000701,-0.000287,-0.000839,0.002296,0.000005,2.831426,0,21315,ok,
127892,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,2,0.003263,0.002952,-0.000553,0.000634,0.004400,0.000019,0.735374,0,21315,ok,
127893,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,3,-0.001437,-0.000470,0.000050,-0.000147,0.001512,0.000002,-2.825468,0,21315,ok,
127894,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,4,0.001378,-0.002210,0.000101,0.000068,0.002605,0.000007,-1.013475,0,21315,ok,
127895,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,5,-0.000871,-0.000614,0.000012,-0.000025,0.001066,0.000001,-2.527207,0,21315,ok,


,channel_id（通道编号）,period_samples（周期采样点数）,frequency_cycles_per_sample（每采样点周频率）,scale（连续核尺度）,scale_weight（SST尺度求积权重）,warmup_samples（预热采样点数）,effective_memory_samples（有效记忆采样点数）,support_future_samples（所需未来采样点数）,group_delay_samples（峰值频率响应滞后点数）,negative_frequency_energy_ratio（连续核负频能量比例）
row（周期配置行）,,,,,,,,,,
0,0,8.0,0.125000,7.639444,0.180900,0,62,62,0.0,8.521578e-17
1,1,16.0,0.062500,15.278889,0.191874,0,123,123,0.0,8.521578e-17
2,2,32.0,0.031250,30.557778,0.135675,0,245,245,0.0,8.521578e-17
3,3,64.0,0.015625,61.115555,0.095937,0,489,489,0.0,8.521578e-17
4,4,128.0,0.007812,122.231111,0.067838,0,978,978,0.0,8.521578e-17
5,5,256.0,0.003906,244.462222,0.015989,0,1956,1956,0.0,8.521578e-17


,sample_id（采样点编号）,sample_at（采样位置时刻）,bar_at（源报价可得时刻）,trading_date（交易日）,sample_side（训练测试侧）,input_value（上游输入值）,channel_id（通道编号）,coefficient_real（系数实部）,coefficient_imag（系数虚部）,derivative_real（导数实部）,derivative_imag（导数虚部）,amplitude（系数幅值）,energy（系数能量）,phase（系数相位弧度）,segment_id（有效段编号）,segment_sample_count（段内有效点数）,transform_status（变换状态）,transform_reason（状态原因）
row（末端变换行）,,,,,,,,,,,,,,,,,,
127888,21315,2026-09-30 14:00:00+08:00,2026-09-30 14:00:00+08:00,2026-09-30,test,-0.000321,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,21314,boundary_incomplete,incomplete_kernel_support
127889,21315,2026-09-30 14:00:00+08:00,2026-09-30 14:00:00+08:00,2026-09-30,test,-0.000321,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,21314,boundary_incomplete,incomplete_kernel_support
127890,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,21315,boundary_incomplete,incomplete_kernel_support
127891,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,21315,boundary_incomplete,incomplete_kernel_support
127892,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,21315,boundary_incomplete,incomplete_kernel_support
127893,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,21315,boundary_incomplete,incomplete_kernel_support
127894,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,21315,boundary_incomplete,incomplete_kernel_support
127895,21316,2026-09-30 15:00:00+08:00,2026-09-30 15:00:00+08:00,2026-09-30,test,-0.000964,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,21315,boundary_incomplete,incomplete_kernel_support


,channel_id（通道编号）,transform_status（变换状态）,transform_reason（状态原因）,通道点数,kernel_type（核类型）
row（真实状态汇总行）,,,,,
0,0,failed,nonfinite_input,1,causal_gammatone
1,0,ok,,21129,causal_gammatone
2,0,warmup,insufficient_history,186,causal_gammatone
3,1,failed,nonfinite_input,1,causal_gammatone
4,1,ok,,20974,causal_gammatone
5,1,warmup,insufficient_history,341,causal_gammatone
6,2,failed,nonfinite_input,1,causal_gammatone
7,2,ok,,20650,causal_gammatone
8,2,warmup,insufficient_history,665,causal_gammatone
